# 📝 Assignment — Chapter 2 — First-Order Logic and Reasoning — agentic lab

This is the **student** notebook. It contains 3 graded tasks.

Work top to bottom. Where you meet a **Task**, replace the `NotImplementedError` with your implementation and run the checks cell that follows. **The assertions in the checks cells are the marking scheme** — when they all pass, the assignment is complete.

Cells outside the tasks are worked examples. Read and run them: they build what the tasks need.

> Worked solutions: [`04_agentic_lab_solution.ipynb`](04_agentic_lab_solution.ipynb)

# Chapter 2 — First-Order Logic and Reasoning
### Notebook 4 · Agentic lab — formalisation, graded semantically

*Book reference: Extends Ch. 2*

An agent that turns English into logic. What makes this lab different from most LLM evaluation: the grader is a **decision procedure**, not a string comparison and not a judge.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch02_toolkit as fol
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
import ch02_agentic as A
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

**By the end of this notebook you can:**

1. Grade a generative task **semantically**, using Chapter 2's own entailment checker instead of string overlap.
2. Turn a countermodel into optimiser feedback.
3. Model **proof search** as an MDP and solve it exactly.
4. Watch GEPA learn the quantifier rules in two distinct stages.

> **Prerequisite:** Chapter 1's agentic lab, which introduced tools, metrics, MDPs, GEPA and skills. Here the task changes and the discipline does not.

## 1. Why this task is unusually well-posed

Most generative tasks are graded by string overlap (crude) or by a judge (expensive, and itself unvalidated). Formalisation has neither problem: two formulas are equivalent when they have the same models, and Notebook 1 built the machinery to check that.

So `forall x (P(x) -> Q(x))` and `~exists x (P(x) & ~Q(x))` both earn full marks, while a formula that merely *looks* similar earns none.

In [ ]:
a = 'forall x (Human(x) -> Mortal(x))'
b = '~exists x (Human(x) & ~Mortal(x))'      # different string, same meaning
c = 'forall x (Human(x) & Mortal(x))'        # similar string, different meaning
print(f'{a}\n  == {b} ?', A.equivalent(fol.parse(a), fol.parse(b)))
print(f'{a}\n  == {c} ?', A.equivalent(fol.parse(a), fol.parse(c)))

## 2. Tools

The agent gets the reasoning services as function tools. Note `check_entailment` returns a **countermodel** on failure — a tool that explains its 'no' is worth far more to an agent than one that merely reports it.

In [ ]:
ctx = A.Ch2Context()
tools = {t.name: t for t in A.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:20s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":20s} {t.description.splitlines()[0]}')

In [ ]:
print(tools['assert_premise'].invoke({'text': 'forall x (Human(x) -> Mortal(x))'}))
print(tools['assert_premise'].invoke({'text': 'Human(Socrates)'}))
print(tools['check_entailment'].invoke({'conclusion': 'Mortal(Socrates)'}))
print(tools['check_entailment'].invoke({'conclusion': 'Mortal(Plato)'}))
print()
print(tools['prove'].invoke({'conclusion': 'Mortal(Socrates)'}))
print('\ntool calls logged:', ctx.log.names())

## 3. The dataset and the semantic metric

Ten statements, split by item. The metric is staged, and the staging is deliberate:

| outcome | score |
|---|---|
| does not parse | 0.00 |
| parses, wrong meaning | 0.25 |
| semantically equivalent | 1.00 |

A flat zero for both failure modes would tell the optimiser only *that* it failed. Partial credit for well-formedness gives it a gradient to climb in two steps — first *be parseable*, then *be right*.

In [ ]:
train, dev = A.build_dataset('train'), A.build_dataset('dev')
print(f'train {len(train)}, dev {len(dev)}')
for e in dev:
    print(f'  {e.id:28s} {e.statement:45s} {e.gold_formula}')

In [ ]:
lm = llm.configure_dspy(A.FOL_RULEBOOK, A.fol_responder)
baseline = A.FormalisationProgram()
example = train[0]
pred = baseline(**example.inputs())
report = A.translation_scorer(example, pred)
print('statement :', example.statement)
print('produced  :', repr(pred.formula))
print('score     :', report.score)
for n in report.notes:
    print('  ', n)

### Feedback carries the countermodel

Once the formula parses, a wrong reading is diagnosed by *name* and witnessed by a model. This is what the GEPA reflection step reads:

In [ ]:
mid = A.FormalisationProgram(
    A.BASELINE_INSTRUCTION + '\n- RULE emit-parseable-formula: answer with the formula alone')
gepa_metric = ev.make_gepa_metric(A.translation_scorer, A.FOL_RULEBOOK)
fb = gepa_metric(example, mid(**example.inputs()))
print('score:', fb.score)
print(fb.feedback)

## 4. Optimising with GEPA — in two stages

The baseline scores **0.0**: it wraps every formula in prose, so nothing parses. Watch the optimiser first learn to emit a bare formula, then learn the quantifier semantics.

In [ ]:
before = ev.evaluate_dataset(baseline, dev, A.translation_scorer)
print('BEFORE:', before['mean_score'], before['violations'])

In [ ]:
reflect = llm.reflection_lm(A.FOL_RULEBOOK, A.fol_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=80, reflection_lm=reflect)
result = opt.compare(A.FormalisationProgram(), tuned, dev, A.translation_scorer)
print(result.report())

In [ ]:
found = A.FOL_RULEBOOK.active_in(result.instruction_after)
print('rules discovered:', sorted(found))
print('rules missed    :', sorted(set(A.FOL_RULEBOOK.ids) - found))

## 5. Proof search as an MDP

Chapter 1's MDP bought information; Chapter 4's built an artefact. This one **searches**.

| | Chapter 2 proof search |
|---|---|
| **S** | the set of clauses derived so far, plus whether a verdict was given |
| **A** | derive a resolvent, or claim entailed / not-entailed |
| **T** | deterministic — resolution is a function of the clauses |
| **R** | −cost per resolution; **+1 for a justified correct verdict**, −1 otherwise |

The word *justified* is doing real work: claiming entailment is only rewarded once the empty clause is actually derived. Guessing right is not the same as proving.

In [ ]:
prem = [fol.parse('forall x (Human(x) -> Mortal(x))'), fol.parse('Human(Socrates)')]
goal = fol.parse('Mortal(Socrates)')
clauses = []
for f in prem + [fol.Not(goal)]:
    clauses += fol.to_cnf_clauses(fol.ground(f, ['Socrates']))

M = A.ProofSearchMDP(clauses, entailed=True, step_cost=0.05)
print('clause universe (base + resolution closure):')
for i, c in enumerate(M.universe):
    print(f'  {i}: ' + ('EMPTY' if not c else '{' + ', '.join(sorted(c)) + '}'))
print(f'\n|S| = {len(M.states())}')

In [ ]:
V, pi = mdp.value_iteration(M)
s0 = M.initial_state()
print(f'V*(s0) = {V[s0]:.3f}\n')
ep = mdp.run_episode(M, mdp.greedy_policy(pi))
for t in ep.transitions:
    print(f'  {str(t.state):16s} {M.describe_action(t.action):34s} r={t.reward:+.2f}')
print(f'\nreturn = {ep.discounted_return():.3f}  '
      f'(1.0 for the verdict, minus {len(ep)-1} resolution steps)')

In [ ]:
print('random policy value:', round(mdp.policy_value(M, mdp.random_policy(), 400), 3))
print('optimal value      :', round(V[s0], 3))
print('\nA random searcher does far worse than nothing: it claims verdicts it\n'
      'has not justified and is penalised. Requiring the empty clause before\n'
      'crediting a claim is what makes the reward reward *proving*.')

### Task 4.1 — Find the budget at which GEPA stops learning

Sweep `max_metric_calls` and record the held-out score and the rules discovered. Report the smallest budget that finds all five rules.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError


**Checks for Task 4.1.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert len(rows) >= 3, 'sweep at least three budgets'
assert [r['budget'] for r in rows] == sorted(r['budget'] for r in rows)
assert all({'budget', 'dev_score', 'n_rules'} <= set(r) for r in rows), (
    'report the budget beside the score -- a score without its budget is uninterpretable')
assert all(0.0 <= r['dev_score'] <= 1.0 and r['n_rules'] >= 0 for r in rows)

### Task 4.2 — Break the semantic grader

Find a predicted formula that is **not** equivalent to the gold formula in general, but that `equivalent(..., max_size=2)` accepts. Then show a larger `max_size` catching it.

> **Hint.** Reuse Exercise 2.2: how large must a domain be before quantifier order can matter at all?

In [ ]:
# YOUR CODE HERE

raise NotImplementedError


**Checks for Task 4.2.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert A.equivalent(guess, gold, max_size=1)
assert not A.equivalent(guess, gold, max_size=2)

### Task 4.3 — Punish unproved verdicts harder

In the proof-search MDP, raise `wrong_verdict_penalty` and find the point at which the optimal policy prefers *not* to answer at all. Discuss what that means for an agent asked a question it cannot settle.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError


**Checks for Task 4.3.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert [r['penalty'] for r in rows] == [0.0, 1.0, 5.0]
assert all({'penalty', 'V*', 'steps', 'verdict'} <= set(r) for r in rows)
# Raising the penalty for an unproved verdict cannot raise the optimal value.
assert rows[-1]['V*'] <= rows[0]['V*'] + 1e-9
# entailed=False, so the correct verdict needs no proof: the agent is right for
# free however harshly wrong verdicts are punished. That asymmetry is the lesson.
assert all('claim' in r['verdict'] for r in rows)

## Chapter 2 in the course arc

| | Ch. 1 | Ch. 2 | Ch. 4 |
|---|---|---|---|
| task | assess an artefact | formalise English | build an axiom |
| MDP | gather evidence | **search for a proof** | construct under constraints |
| grader | labels + judge | **a decision procedure** | labels + profile table |
| GEPA learns | reporting discipline | quantifier semantics | OWL 2 profile limits |

Chapter 2's distinctive contribution is the grader: when your task has a decision procedure, use it — it is cheaper, sharper and more honest than any judge. Chapter 3 asks what happens when you *design a language* so that such a procedure always exists.